# 第6章：自注意力机制 (Self-Attention)

> "Attention is All You Need——自注意力是Transformer的核心引擎，是现代大模型的基础构件。"

## 本章知识导图

```
自注意力机制 (Self-Attention)
│
├── 6.1 输入是向量序列的情况
│   ├── 类型1：输入输出数量相同（词性标注、每个词→标签）
│   ├── 类型2：序列→单个标签（情感分析、整句→正面/负面）
│   └── 类型3：Seq2Seq（机器翻译、输入N词→输出M词）
│
├── 6.2 自注意力核心公式
│   └── Attention(Q,K,V) = softmax(QK^T/√d_k)·V
│
├── 6.3 逐步理解
│   ├── Q=X·W_Q, K=X·W_K, V=X·W_V (三个线性投影)
│   ├── QK^T：计算每对位置的相关度
│   ├── /√d_k：缩放防止Softmax饱和
│   ├── Softmax：归一化为概率
│   └── ×V：按注意力权重加权求和
│
├── 6.4 多头注意力：多组Q/K/V并行
├── 6.5 位置编码：弥补自注意力缺失的位置信息
├── 6.6 截断自注意力：限制局部窗口
├── 6.7 自注意力 vs CNN：CNN是局部自注意力的特例
└── 6.8 自注意力 vs RNN：可完全并行！
```

## 6.0 Q/K/V矩阵的完整数值走查：一个玩具示例

### 为什么叫Query/Key/Value？

这三个名字来自**信息检索**的隐喻：

- **Query (查询)**：你想查什么？("我对这个问题关心什么？")
- **Key (键)**：每个内容的标签("我的内容是什么？")
- **Value (值)**：每个内容的实际信息("我能提供什么信息？")

操作：用Query去匹配Key → 匹配度越高，对应的Value被取出的越多。

### 具体数值走查（2个词，4维embedding）

假设我们处理一个2词的句子"The cat"，embedding维度d=4。

**Step 1: 输入X**
$$X = \begin{bmatrix} 
1.0 & 0.0 & 1.0 & 0.0 \\  % token "The"
0.0 & 1.0 & 0.0 & 1.0     % token "cat"
\end{bmatrix} \quad \text{shape: } (2 \times 4)$$

**Step 2: 可学习的投影矩阵**（简化为已知值用于演示）

$$W_Q = \begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix} \quad
W_K = \begin{bmatrix}
0 & 1 & 0 & 0 \\
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 1 \\
0 & 0 & 1 & 0
\end{bmatrix} \quad
W_V = \begin{bmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1
\end{bmatrix}$$

**Step 3: 计算Q, K, V**

$$Q = X W_Q = \begin{bmatrix} 1.0 & 0.0 & 1.0 & 0.0 \\ 0.0 & 1.0 & 0.0 & 1.0 \end{bmatrix}$$

$$K = X W_K = \begin{bmatrix} 0.0 & 1.0 & 0.0 & 1.0 \\ 1.0 & 0.0 & 1.0 & 0.0 \end{bmatrix}$$

$$V = X W_V = \begin{bmatrix} 1.0 & 0.0 & 1.0 & 0.0 \\ 0.0 & 1.0 & 0.0 & 1.0 \end{bmatrix}$$

**Step 4: 计算注意力分数 $QK^T$**
$$S = QK^T = \begin{bmatrix} 
1.0 & 0.0 & 1.0 & 0.0 \\
0.0 & 1.0 & 0.0 & 1.0
\end{bmatrix} 
\begin{bmatrix}
0.0 & 1.0 \\
1.0 & 0.0 \\
0.0 & 1.0 \\
1.0 & 0.0
\end{bmatrix} = \begin{bmatrix}
0.0 & 2.0 \\
1.0 & 0.0
\end{bmatrix}$$

解读：$S_{11}=0.0$（"The"对"The"的关注度），$S_{12}=2.0$（"The"对"cat"的关注度 — 高！），$S_{21}=1.0$（"cat"对"The"的关注度），$S_{22}=0.0$（"cat"对"cat"的关注度）。

**Step 5: 缩放** $\sqrt{d_k} = \sqrt{4} = 2$
$$S_{scaled} = S / 2 = \begin{bmatrix} 0.0 & 1.0 \\ 0.5 & 0.0 \end{bmatrix}$$

**Step 6: Softmax（按行）**
$$A = \text{softmax}\left(\begin{bmatrix} 0.0 & 1.0 \\ 0.5 & 0.0 \end{bmatrix}\right) = \begin{bmatrix} 0.269 & 0.731 \\ 0.622 & 0.378 \end{bmatrix}$$

第一行："The"把73.1%的注意力放在"cat"上，26.9%放在自己身上。
第二行："cat"把62.2%的注意力放在"The"上，37.8%放在自己身上。

**Step 7: 加权求和 — 最终输出**
$$\text{Output} = A \cdot V = \begin{bmatrix} 0.269 & 0.731 \\ 0.622 & 0.378 \end{bmatrix} \begin{bmatrix} 1.0 & 0.0 & 1.0 & 0.0 \\ 0.0 & 1.0 & 0.0 & 1.0 \end{bmatrix}$$

$$= \begin{bmatrix} 0.269 & 0.731 & 0.269 & 0.731 \\ 0.622 & 0.378 & 0.622 & 0.378 \end{bmatrix}$$

观察输出第一行："The"的新表示不仅包含自己的信息，还融合了大量来自"cat"的信息（第二列0.731），这体现了"跨位置的信息聚合"。

> **核心洞察：** 自注意力后的每个位置的向量，是所有位置V的加权和。权重由Q和K的点积相似度决定。这就是"每个位置直接和其他所有位置通信"的机制。</cell>


## 6.1.1 缩放因子的数学原理：为什么是 √d_k？

### 问题的提出

为什么自注意力公式是 $\text{softmax}(QK^T/\sqrt{d_k})$ 而不是 $\text{softmax}(QK^T)$？去掉这个缩放会怎样？

### 点积的统计特性

假设Q和K的每个元素独立同分布，均值为0，方差为1。那么点积的统计特性如下：

$$q \cdot k = \sum_{i=1}^{d_k} q_i k_i$$

- $\mathbb{E}[q_i k_i] = 0$（独立，均值都为0）
- $\text{Var}(q_i k_i) = \text{Var}(q_i) \cdot \text{Var}(k_i) = 1 \times 1 = 1$
- $\text{Var}(q \cdot k) = \sum_{i=1}^{d_k} \text{Var}(q_i k_i) = d_k$

**结论：点积的方差 = $d_k$！** 当$d_k$很大时（如64或128），点积值的范围在±几十左右。

### 为什么大方差有害？

Softmax函数的梯度特性：
- 当输入值差异较小时 → Softmax比较平滑 → 梯度正常
- 当输入值差异很大时 → Softmax接近one-hot → 梯度趋近于0（饱和）

```
Softmax饱和示例（无缩放）:
input: [0.5, 65.0, -65.0] → softmax → [~0, ~1, ~0]   ← 梯度≈0
input: [0.5, 2.0, -2.0]   → softmax → [0.15, 0.67, 0.18]  ← 梯度正常

Softmax饱和示例（有缩放√d_k=8）:
scaled: [0.06, 8.12, -8.12] → softmax → [0.00, 1.00, 0.00] ← 仍然有梯度
wait, that's still saturated...
实际中weights初始化为小值，所以实际点积不会这么大
关键是保持梯度在训练初期也足够：∂softmax/∂x = softmax_i*(1-softmax_i) //还是没怎么想明白
```

### 渐进推导

没有缩放的注意力：$A_{ij} = \frac{\exp(Q_i \cdot K_j)}{\sum_k \exp(Q_i \cdot K_k)}$

当$d_k$增大，$Q_i \cdot K_j$的方差异增大 → Softmax的分布变得极端 → 少数位置获得几乎全部注意力 → 梯度消失 → 训练困难。

引入$1/\sqrt{d_k}$后：
$$A_{ij} = \frac{\exp(Q_i \cdot K_j / \sqrt{d_k})}{\sum_k \exp(Q_i \cdot K_k / \sqrt{d_k})}$$

缩放后点积方差还原为1，Softmax分布保持平滑 → 梯度充足 → 训练稳定。

### 数值验证

```python
import torch
# d_k=64 (大维度)
q = torch.randn(64)  # 均值0, 方差1
k = torch.randn(64)
dot = torch.dot(q, k)
print(f"d_k=64, 点积={dot:.2f}, 缩放后={dot/8:.2f}")  # 典型值在±16, 缩放后±2

# d_k=512 (更大的维度)
q = torch.randn(512)
k = torch.randn(512)
dot = torch.dot(q, k)
print(f"d_k=512, 点积={dot:.2f}, 缩放后={dot/22.6:.2f}") # 典型值在±45, 缩放后±2
```

> **关键结论：** $1/\sqrt{d_k}$ 这个缩放因子不是随意的——它统计上抵消了点积方差随维度线性增长的问题，确保Softmax不饱和，梯度流动顺畅。这是Transformer论文中最精妙的设计细节之一。</cell>


## 6.1 为什么需要自注意力？

### RNN的问题
RNN要处理"The animal didn't cross the street because it was too tired"中"it"指什么的问题，但它有几个致命弱点：
1. **串行处理**：必须一步一步来，不能并行
2. **长程依赖差**：远距离的词之间的信息传递经过很多步，容易丢失
3. **计算效率低**：GPU并行能力被浪费

### 自注意力的解决方案
让序列中的**每个位置直接和所有其他位置建立联系**——不需要通过中间步骤传递。

> **图书馆比喻：**
> - Q (Query/查询)：你想知道什么？("it指什么？")
> - K (Key/键)：每个词的索引标签("我是animal")
> - V (Value/值)：每个词的实际信息内容
> 你说出Query→匹配关键词Key→拿到对应的Value。整个过程对所有位置并行执行！

## 6.2.1 多头注意力详解：每个头看不同的关系

### 具体句子演示

考虑句子："*昨天,小明去了北京,他参观了故宫*"

8个注意力头可能学到：

| 头编号 | 可能学到的关系 | 注意力模式 |
|--------|---------------|-----------|
| Head 1 | **语法：主谓关系** | "小明" ↔ "去"、"他" ↔ "参观" |
| Head 2 | **指代消解** | "他" → "小明"（强注意力） |
| Head 3 | **时间修饰** | "昨天" ↔ "去"、"参观" |
| Head 4 | **地点修饰** | "北京" ↔ "故宫"、"去" |
| Head 5 | **相邻词依赖** | 每个词关注其左邻右舍 |
| Head 6 | **全局位置** | 所有词均匀关注开头几个词([CLS]-like) |
| Head 7 | **语义相似** | "小明" ↔ "他"（语义相近） |
| Head 8 | **冗余/无用** | 可能什么都不学（"dead head"） |

### 多头注意力的并行计算

**单头**参数量：$3 \times d_{model} \times d_k$ (三组W_Q, W_K, W_V投影矩阵)

**多头**（h个头，每头$d_k = d_{model}/h$）：
$$P_{multi} = h \times 3 \times d_{model} \times (d_{model}/h) = 3 \times d_{model}^2$$

**关键发现：多头注意力的总参数量 = 单头（当$d_k = d_{model}$）的参数量的 $1/h$！**

等等，实际上是相等的：$h \times 3 \times d_{model} \times d_k = h \times 3 \times d_{model} \times (d_{model}/h) = 3 \times d_{model}^2$。

**所以多头和单头参数总量相同，但多头有更多"视角"！** 这就是为什么多头更好的原因——用同样的参数预算换来了多样化的注意力模式。

### 代码演示：头维度的影响

```python
# 同一embed_dim，不同头数的对比
# embed_dim=512, num_heads=1 → 每头512维（单视角）
# embed_dim=512, num_heads=8 → 每头64维（8视角）

# 参数量完全相同！
# num_heads=1: 3×512×512 = 786,432
# num_heads=8: 8×3×512×64 = 786,432  ← 一样！
```

> **直觉：** 一个512维的视角 vs 8个64维的视角。8个窄视角能捕捉到单一宽视角可能忽略的细节关系，因为不同的头有不同的初始化和不同的梯度更新路径，自然分化出不同的注意力模式。</cell>


## 6.2 自注意力的完整数学推导

### Step 1：从输入生成Q、K、V

设输入序列$X \in \mathbb{R}^{L \times d}$（L个token，每个d维），乘以三个可学习的权重矩阵：

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

其中$W_Q, W_K, W_V \in \mathbb{R}^{d \times d_k}$（通常$d_k = d$）。

### Step 2：计算注意力分数

$$\text{Scores} = QK^T \in \mathbb{R}^{L \times L}$$

$\text{Scores}_{ij}$ = 第i个位置对第j个位置的"关注度"。用**点积**来衡量：Q和K越相似，点积越大，关注度越高。

### Step 3：缩放 (关键步骤！)

$$\text{Scores}_{\text{scaled}} = \frac{QK^T}{\sqrt{d_k}}$$

**为什么需要$\sqrt{d_k}$？** 当$d_k$很大时（如512），点积$QK^T$的值可能很大。输入到Softmax的数值过大会使梯度趋近于0（Softmax饱和）。缩放后保持在合理范围。

### Step 4：Softmax归一化

$$A = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)$$

A的每一行是一个概率分布（和为1），表示这个位置对所有位置的注意力权重。

### Step 5：加权求和

$$\text{Output} = A \cdot V$$

每个输出位置 = 所有输入位置的V的加权和，权重由注意力分布决定。

> **最终结果：** 经过自注意力后，每个位置的输出不再只包含自己的信息，而是融合了**整个序列中所有相关位置的信息**。比如"it"的输出向量会被"animal"的信息大幅"增强"。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ============================================================
# 从零实现多头注意力 (Multi-Head Attention)
# 不使用 nn.MultiheadAttention，完全手工
# ============================================================
class MultiHeadAttention(nn.Module):
    """从零实现的多头注意力机制"""
    def __init__(self, d_model=512, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # 三个投影矩阵（将总维度一次性投影，然后拆分）
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # 输出投影
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)
    
    def forward(self, query, key, value, mask=None):
        """
        query: (batch, seq_len, d_model)
        key:   (batch, seq_len, d_model)
        value: (batch, seq_len, d_model)
        """
        batch_size = query.size(0)
        
        # 1. 线性投影并拆分为多头
        # (B, L, d_model) → (B, L, num_heads, d_k) → (B, num_heads, L, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # 2. 计算注意力分数
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (B, h, L_q, L_k)
        
        # 3. 掩码处理（因果掩码/padding掩码）
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # 4. Softmax + Dropout
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # 5. 加权求和
        out = torch.matmul(attn_weights, V)  # (B, h, L_q, d_k)
        
        # 6. 合并多头并投影
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        out = self.W_o(out)
        
        return out, attn_weights

# ============================================================
# 测试：自注意力
# ============================================================
d_model = 512
num_heads = 8
batch_size = 2
seq_len = 10

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
x = torch.randn(batch_size, seq_len, d_model)

# 自注意力
out, attn = mha(x, x, x)
print(f"输入: {x.shape}")
print(f"输出: {out.shape}")
print(f"注意力权重: {attn.shape}  (batch, num_heads, seq_len, seq_len)")
print(f"\n参数量统计:")
total_params = sum(p.numel() for p in mha.parameters())
print(f"  总参数: {total_params:,}")
print(f"  理论值: 4 × d_model² = {4 * d_model * d_model:,}")

# 交叉注意力（常见于解码器）
y = torch.randn(batch_size, 15, d_model)  # 来自编码器的输出
out_cross, attn_cross = mha(x, y, y)
print(f"\n交叉注意力: query={x.shape[1]}个token, key/value={y.shape[1]}个token")
print(f"输出: {out_cross.shape}")
print(f"注意力权重: {attn_cross.shape}")

# 因果掩码测试
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
print(f"\n因果掩码形状: {causal_mask.shape}")
print(f"掩码示例（5×5）:")
print(causal_mask[0, 0, :5, :5].int())
out_masked, attn_masked = mha(x, x, x, mask=causal_mask)
print(f"\n带因果掩码的输出: {out_masked.shape}")
# 验证上三角权重为0
print(f"第一个头、第一个查询对后续位置的注意力: {attn_masked[0, 0, 0, 1:5].tolist()}")
print(f"(这些值应该都是0，因为因果掩码阻止'看未来')")

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# 注意力权重热力图可视化
# ============================================================
print("=" * 60)
print("注意力权重可视化")
print("=" * 60)

# 创建一个简单的句子
sentence = ["我", "爱", "深度", "学习", "因为", "它", "很", "强大"]
seq_len = len(sentence)

# 模拟注意力权重（使用实际计算）
d_model = 64
# 为每个词创建随机embedding，但让语义相关的词更相似
torch.manual_seed(42)
embeddings = torch.randn(1, seq_len, d_model)

# 使用单头自注意力
W_q = torch.randn(d_model, d_model) * 0.1
W_k = torch.randn(d_model, d_model) * 0.1
W_v = torch.randn(d_model, d_model) * 0.1

Q = embeddings @ W_q
K = embeddings @ W_k
V = embeddings @ W_v

scores = Q @ K.transpose(-2, -1) / (d_model ** 0.5)
attn_weights = F.softmax(scores, dim=-1)[0]  # (seq_len, seq_len)

print(f"注意力权重矩阵形状: {attn_weights.shape}")
print(f"每行的和 = {attn_weights.sum(dim=-1).tolist()}  (Softmax保证为1)")

# 可视化热力图
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(attn_weights.numpy(), cmap='YlOrRd', aspect='auto')

# 标注
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(sentence, fontsize=12)
ax.set_yticklabels(sentence, fontsize=12)
ax.set_xlabel('Key (被关注的位置)', fontsize=13)
ax.set_ylabel('Query (发起关注的位置)', fontsize=13)
ax.set_title('自注意力权重热力图\n颜色越深 = 注意力越强', fontsize=14)

# 添加数值标注
for i in range(seq_len):
    for j in range(seq_len):
        text = ax.text(j, i, f'{attn_weights[i, j]:.2f}',
                       ha="center", va="center", 
                       color="white" if attn_weights[i, j] > 0.3 else "black",
                       fontsize=10)

plt.colorbar(im, ax=ax, label='注意力权重')
plt.tight_layout()
plt.show()

print(f"\n=== 注意力模式解读 ===")
# 找出每行的最大值
for i in range(seq_len):
    max_j = attn_weights[i].argmax().item()
    print(f"'{sentence[i]}' 最关注 '{sentence[max_j]}' (权重={attn_weights[i, max_j]:.3f})")

print(f"\n=== 多头注意力的热力图含义 ===")
print("不同颜色深度的行/列表示了token间的依赖关系")
print("对角线通常较强（每个词关注自己）")
print("强跨词注意力 = 模型认为这两个词有重要关系")
print("在实际模型中，不同注意力头会呈现不同的关注模式")
print("通常某些头擅长语法关系，某些擅长语义关系，某些关注位置")

# 演示因果掩码（下三角）用于解码器
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
masked_scores = scores + (1 - causal_mask) * (-1e9)
masked_attn = F.softmax(masked_scores, dim=-1)[0]

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(masked_attn.numpy(), cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(sentence, fontsize=12)
ax.set_yticklabels(sentence, fontsize=12)
ax.set_xlabel('Key', fontsize=13)
ax.set_ylabel('Query', fontsize=13)
ax.set_title('带因果掩码的注意力（解码器用）\n上三角全为0：不能"偷看"未来token', fontsize=14)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()</cell>


## 6.5 截断自注意力 (Truncated Self-Attention)

### 为什么需要截断？

标准自注意力的复杂度是 $O(L^2)$ — 每个位置关注所有其他位置。当序列长度L=10,000时，注意力矩阵是10,000×10,000 = 1亿个元素！

这对长文本（如整本书）和长视频完全不可行。

### 截断自注意力的思想

不让每个token关注所有token，而是只关注局部窗口内的token：

```
标准自注意力:              截断自注意力 (窗口大小w=3):
┌─────────────────┐        ┌─────────────────┐
│· · · · · · · · ·│        │· · · · · · · · ·│
│· · · · · · · · ·│        │· · · · · · · · ·│
│· · · · · · · · ·│        │· · · · · · · · ·│
│· · · · · · · · ·│        │· · █ █ █ · · · ·│  ← 每个token只关注
│· · · · · · · · ·│        │· · · █ █ █ · · ·│    窗口内的token
│· · · · · · · · ·│        │· · · · █ █ █ · ·│
│· · · · · · · · ·│        │· · · · · █ █ █ ·│
│· · · · · · · · ·│        │· · · · · · · · ·│
│· · · · · · · · ·│        │· · · · · · · · ·│
└─────────────────┘        └─────────────────┘
复杂度: O(L²)               复杂度: O(L × w)
```

### 三种截断策略

**1. 滑动窗口 (Sliding Window)**
每个token关注前后各w/2个token。简单高效，但忽略了远距离关系。

**2. 扩张窗口 (Dilated Window)**
每隔几个位置取一个，扩大感受野而不增加计算量。类似于扩张卷积(Dilated Convolution)。

```
标准窗口(间隔=1):  扩张窗口(间隔=2):
位置i关注:           位置i关注:
[i-2, i-1, i, i+1, i+2]  [i-4, i-2, i, i+2, i+4]
```

**3. 全局+局部 (Global+Local)**
大部分层用局部窗口，少数层用全局注意力。Longformer使用这种策略：某些特殊token（如[CLS]）使用全局注意力。

### 截断策略对比

| 方法 | 复杂度 | 优点 | 缺点 |
|------|--------|------|------|
| 全注意力 | $O(L^2)$ | 完美覆盖所有依赖 | 长序列不可行 |
| 滑动窗口 | $O(L \cdot w)$ | 简单高效 | 丢失远距离依赖 |
| 扩张窗口 | $O(L \cdot w)$ | 扩大感受野 | 可能跳过关键信息 |
| 全局+局部 | $O(L \cdot w + G \cdot L)$ | 平衡效率与覆盖 | 需要选择全局token |
| 稀疏注意力(随机) | $O(L \cdot w)$ | 数学保证收敛 | 实现复杂 |

> **实践中的做法：** Longformer, BigBird, Reformer都使用某种形式的稀疏/截断注意力。普通Transformer通常限制最大序列长度到512或1024。对于更长序列，使用分块(chunking)、摘要(summarization)或层次化注意力。</cell>


## 6.6 自注意力的计算复杂度分析与优化方向

### 时间复杂度

标准自注意力的计算流程（序列长度L，维度d）：

| 操作 | 复杂度 | 当L=1024, d=768时 |
|------|--------|-------------------|
| $Q = XW_Q$ | $O(L \cdot d^2)$ | 1024 × 768² ≈ 6亿 |
| $K = XW_K$ | $O(L \cdot d^2)$ | 同上 |
| $V = XW_V$ | $O(L \cdot d^2)$ | 同上 |
| $S = QK^T$ | $O(L^2 \cdot d)$ | 1024² × 768 ≈ 8亿 |
| $\text{Softmax}(S)$ | $O(L^2)$ | 1024² ≈ 1M |
| $O = AV$ | $O(L^2 \cdot d)$ | 同上 |
| $O_{out} = OW_O$ | $O(L \cdot d^2)$ | 同上 |

- 当$L < d$（短序列）时：$O(L \cdot d^2)$ 主导（投影矩阵乘法）
- 当$L > d$（长序列）时：**$O(L^2 \cdot d)$ 主导**（注意力矩阵）

### 内存复杂度

注意力矩阵存储：$L \times L = L^2$ 个float16/float32：
- L=512: 512²×4 = 1MB（可接受）
- L=1024: 1024²×4 = 4MB（可接受）
- L=4096: 4096²×4 = 64MB（较大）
- L=32768: 32768²×4 = 4GB（需要优化）
- L=1M: 1M²×4 = 4TB（完全不可行）

### 为什么长序列是瓶颈？

这不是理论问题 —— 每个LLM应用都面临这个挑战：
1. **文档问答**：一篇论文可能有20K tokens
2. **代码补全**：一个文件可能有5K tokens
3. **多轮对话**：历史可能有10K tokens
4. **基因组分析**：DNA序列可能有百万级碱基对

### 主要优化方向

| 方法 | 代表 | 核心思想 | 复杂度 |
|------|------|----------|--------|
| FlashAttention | Dao et al. 2022 | IO感知的tiling + recomputation | 内存$O(N)$ |
| 线性注意力 | Katharopoulos 2020 | $\phi(Q)\phi(K)^TV$（不用显式$L^2$矩阵） | $O(L \cdot d^2)$ |
| 稀疏注意力 | Longformer, BigBird | 滑动窗口+全局token | $O(L \cdot w)$ |
| 低秩近似 | Linformer | 将K/V投影到固定维度k | $O(L \cdot k \cdot d)$ |
| KV Cache | 标准推理优化 | 缓存已生成的K/V，不重复计算 | 推理加速 |
| PagedAttention | vLLM | 分页管理KV cache | 内存效率 |
| Ring Attention | 分布式 | 序列维度切分到多个GPU | 扩展性 |

> **最重要的实际进展：FlashAttention** — 通过IO感知的分块计算（在SRAM中完成注意力计算的大部分操作，只将最终结果写回HBM），在不改变数学结果的前提下，内存从$O(L^2)$降到$O(N)$。PyTorch 2.0+通过`torch.nn.functional.scaled_dot_product_attention`原生支持FlashAttention。</cell>


## 6.7 自注意力 vs CNN vs RNN 终极对比

### 三种架构的本质差异

| 维度 | CNN | RNN/LSTM | 自注意力 |
|------|-----|----------|----------|
| **核心操作** | 卷积(局部加权求和) | 循环(状态传递) | 注意力(全局加权求和) |
| **连接模式** | 局部($K \times K$窗口) | 序列(一步接一步) | 全局(任意两位置之间) |
| **信息流动** | 层→层(前向)，无时间维度 | 时间步→时间步(串行) | 直接(任意距离) |
| **参数共享** | 空间维度(平移等变) | 时间维度(BPTT) | 无(但投影矩阵共享) |
| **并行性** | ✅ 高(各位置独立) | ❌ 低(必须串行) | ✅ 高(矩阵运算) |
| **计算复杂度** | $O(K^2 \cdot C_{in} \cdot C_{out} \cdot HW)$ | $O(T \cdot d^2)$ | $O(L^2 \cdot d)$ |
| **序列长度依赖** | 无(固定输入尺寸) | 线性($O(T)$) | **二次($O(L^2)$)** |
| **感受野** | 局部 → 逐渐扩大(堆层) | 全局 → 但逐渐衰减 | 全局(一层即达) |
| **归纳偏置** | 平移等变性(强) | 时序因果性(强) | 无(数据驱动,弱) |
| **位置信息** | 天然(像素坐标) | 天然(时间顺序) | 需要显式编码(PE) |

### 何时用什么？

| 任务类型 | 推荐架构 | 原因 |
|----------|----------|------|
| 图像分类/检测 | CNN / ViT | 图像有强2D空间结构 |
| 图像分类(大数据) | ViT (自注意力) | 足够数据时弱归纳偏置+全局感受野 > 强偏置 |
| 短文本(<512 tokens) | Transformer | 完全并行 + 全局依赖 |
| 长文本(>10K tokens) | 稀疏Transformer/状态空间模型 | O(L²)不可行 |
| 实时语音识别 | CNN/LSTM或流式Transformer | 延迟低，逐帧处理 |
| 时间序列(小数据) | LSTM/CNN | Transformer需要大量数据 |
| 蛋白质结构 | CNN(1D/2D) 或 GNN | 3D结构天然适合图表示 |
| 代码生成 | Transformer(自回归) | 长期依赖+并行训练 |

### CNN作为自注意力的特例

当我们将注意力范围限制在局部窗口($K \times K$)且权重不随输入内容变化时：

$$\text{Attention}_{\text{local}} = \text{softmax}(QK^T + M) \cdot V$$

其中M是掩码矩阵（窗口外的位置设为$-\infty$）。如果再加上权重变为固定卷积核，自注意力就退化成了CNN。

**反过来**：自注意力 = 动态权重的、全局感受野的"卷积"。CNN学到的是"无论输入是什么，这个位置用这个核"，自注意力学到的是"根据输入内容，动态决定关注哪里"。

> **最终结论：** 不存在"最好"的架构。CNN的强归纳偏置在小数据时是优势（学得更快），大数据时是瓶颈（限制了能学到的模式）。Transformer的灵活性在大数据时是优势，小数据时需要大量的数据增强和正则化。这就是为什么现代视觉模型在数据量足够时从CNN转向ViT，但移动端和医疗影像等小数据场景CNN仍是主流。</cell>


## 6.8 本章知识总结与思考题

### 核心公式

**自注意力的完整公式：**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**多头注意力：**
$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W_O$$
$$\text{head}_i = \text{Attention}(QW_{Q_i}, KW_{K_i}, VW_{V_i})$$

### 核心知识回顾

1. **Q/K/V**：Q查询信息，K标记信息，V承载信息。Query匹配Key得到权重，权重作用于Value
2. **点积作为相似度**：Q和K的点积衡量两个位置的"关联程度"
3. **$\sqrt{d_k}$是关键**：抵消点积方差随维度线性增长的问题，防止Softmax饱和
4. **多头 = 多视角**：同参数预算下获得更丰富的注意力模式
5. **$O(L^2)$**：自注意力的计算和内存复杂度 — 长序列最大的挑战
6. **因果掩码**：解码器中上三角全为$-\infty$，防止"偷看"未来
7. **交叉注意力**：Q来自一方，K/V来自另一方 — 编码器-解码器的桥梁
8. **自注意力 vs CNN**：自注意力 = 动态权重的、全局感受野的广义卷积

### 思考题

1. 如果去掉$\sqrt{d_k}$缩放，对于$d_k=64$的注意力头，在训练早期会遇到什么问题？用数值说明。
2. 多头注意力中，如果8个头学会了完全相同的注意力模式，这是一个问题吗？为什么实践中很少发生？
3. 为什么交叉注意力用Q来自解码器、K/V来自编码器，而不是反过来？
4. 对于一个100个token的序列，自注意力的注意力矩阵大小是多少？如果使用窗口大小10的截断注意力呢？
5. 为什么现代LLM（如GPT-4）都使用因果掩码（左到右），而BERT不使用？

### 延伸阅读
- "Attention Is All You Need" (Vaswani et al., 2017) — Transformer原始论文
- "FlashAttention: Fast and Memory-Efficient Exact Attention" (Dao et al., 2022)
- "Longformer: The Long-Document Transformer" (Beltagy et al., 2020)
- "Efficient Transformers: A Survey" (Tay et al., 2020) — 效率优化的全面综述</cell>


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
    """从零实现单头自注意力"""
    def __init__(self, embed_dim):
        super().__init__()
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.scale = embed_dim ** 0.5

    def forward(self, x):
        Q = self.W_q(x)  # (B, L, d)
        K = self.W_k(x)
        V = self.W_v(x)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)
        return output, attn_weights

# 演示
sa = SelfAttention(embed_dim=8)
x = torch.randn(2, 5, 8)  # batch=2, seq_len=5, feature=8
out, attn = sa(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention matrix: {attn.shape}  (每个token对每个token的注意力)")
print(f"\n第一个样本的注意力矩阵:\n{attn[0].detach().numpy()}")
print(f"\n每行的和 = {attn[0].sum(dim=1)}  (Softmax保证每行和为1)")

# 展示缩放因子的重要性
print(f"\n缩放因子 sqrt(d) = {sa.scale:.2f}")
print("不缩放：QK^T值可能>100 → Softmax接近one-hot → 梯度≈0")
print("缩放后：QK^T值保持在合理范围 → Softmax平滑 → 梯度充足")

## 6.9 (补充) 点积注意力 vs 加法注意力：两种相似度计算方式

### 历史上的两种注意力

在Vaswani等人的Transformer(2017)使用点积注意力之前，Bahdanau等人(2015)最早在NMT中使用的注意力的形式是加法注意力(Additive Attention)。

**加法注意力 (Bahdanau Attention, 2015):**
$$\text{score}(q, k) = v^T \tanh(W_1 q + W_2 k)$$

**点积注意力 (Transformer, 2017):**
$$\text{score}(q, k) = q^T k$$

### 两种方式的理论比较

| 维度 | 加法注意力 | 点积注意力 |
|------|-----------|-----------|
| **参数** | 有 $(W_1, W_2, v)$ | 无参数！(直接用q,k) |
| **计算** | 需两次矩阵乘法+tanh+一次点积 | 一次点积 |
| **速度** | 较慢 | **更快**(纯矩阵乘法) |
| **GPU亲和** | 一般 | **极好**(matmul高度优化) |
| **理论基础** | 前馈网络近似任意相似度函数 | Q和K在同一空间时的自然度量 |
| **可扩展性** | 难以扩展到多头 | 自然扩展 |

### 为什么Transformer选择了点积？

1. **速度**：矩阵乘法是现代GPU的"第一语言"
2. **简洁**：不需要额外的学习参数（$W_1, W_2, v$）
3. **可扩展**：容易扩展到多头——只需把维度拆分

> **关键洞察：** 加法注意力理论上更灵活（可用前馈网络逼近任意相似度函数），但点积注意力在实践中足够好且快得多。这是一个"好的理论" vs "好的工程"的完美例子 —— 点积注意力赢得了工程上的胜利。

### 现代注意力的变体

随着研究的深入，线性注意力(Linear Attention)也试图将复杂度从$O(L^2)$降到$O(L)$：

$$\text{LinearAttention}(Q, K, V) = \frac{\phi(Q)(\phi(K)^T V)}{\phi(Q)(\phi(K)^T \mathbf{1})}$$

其中$\phi$是一个非线性特征映射（如$\phi(x) = \text{elu}(x) + 1$）。通过改变计算顺序（先算$K^T V$再算$Q$点积），避免了显式存储$L^2$注意力矩阵。这是Performer(2020)等高效Transformer的关键技术。</cell>


## 6.3 多头注意力 (Multi-Head Attention)

### 为什么需要多头？
单头注意力只能捕捉一种"关系"。但语言中有多种不同层次的关系：
- 语法关系（主谓宾）
- 指代关系（代词→先行词）
- 语义相似（近义词）
- 位置关系（相邻词）

多头注意力的做法：将嵌入维度分成h个"头"（如512维分8头，每头64维），每个头独立计算注意力，最后拼接：

$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W_O$$
$$\text{head}_i = \text{Attention}(QW_{Qi}, KW_{Ki}, VW_{Vi})$$

每个头的$W_{Qi}, W_{Ki}, W_{Vi}$不同 → 投影到不同的子空间 → 学到不同类型的注意力模式。

In [ ]:
# PyTorch内置多头注意力
mha = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)
x = torch.randn(2, 10, 512)  # 2个序列，各10个token，每个512维

# 自注意力
out_self, attn_self = mha(x, x, x)
print(f"Self-Attention output: {out_self.shape}")
print(f"Attention weights: {attn_self.shape}  (batch, seq, seq)")

# 交叉注意力 (Q≠K,V)
encoder_out = torch.randn(2, 15, 512)  # 编码器输出15个token
decoder_in = torch.randn(2, 8, 512)    # 解码器输入8个token
out_cross, attn_cross = mha(decoder_in, encoder_out, encoder_out)
print(f"\nCross-Attention: decoder({decoder_in.shape[1]} tokens) attends to encoder({encoder_out.shape[1]} tokens)")
print(f"Output: {out_cross.shape}")

## 6.4 位置编码

### 自注意力的一个"盲区"
自注意力对所有位置"一视同仁"——如果交换两个字的位置，输出也会相应交换。它本身不知道谁在前谁在后。

但位置很重要！"狗咬人" ≠ "人咬狗"。

### 解决方案：位置编码
在输入向量上加上一个表示位置的向量。两种方式：

**1. 正弦位置编码 (原始Transformer)：**
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$
无需学习，可以外推到训练时没见过的长度。

**2. 可学习位置编码 (BERT/GPT)：**
直接用一个Embedding层学习每个位置的向量。需要预设最大长度。

```python
pos_embedding = nn.Embedding(max_len, d_model)
x = x + pos_embedding(torch.arange(seq_len, device=x.device))
```

此外还有**相对位置编码**、**旋转位置编码(RoPE)**等改进方案。

## 6.5 自注意力 vs CNN vs RNN

| 特性 | CNN | RNN | Self-Attention |
|------|-----|-----|----------------|
| 感受野 | 局部(固定) | 全局(但衰减) | 全局(直接) |
| 并行性 | 好 | 差(串行) | **好(完全并行)** |
| 长程依赖 | 需多层堆叠 | 梯度消失 | **直接访问** |
| 参数量 | $K^2 C_{in} C_{out}$ | O(d²) | O(d²) |
| 位置信息 | 天然具备 | 天然具备 | 需要位置编码 |

> CNN可以看作自注意力的特例——当注意力范围限制在局部邻域时。

## 本章核心收获
1. 自注意力=Q查询K→权重×V→聚合全局上下文
2. 缩放因子√d_k是**关键**——防止Softmax梯度消失
3. 多头注意力=多个子空间并行→捕捉多样关系
4. 位置编码补全了自注意力缺失的位置信息
5. 自注意力比RNN好在：完全并行、长程依赖好
6. 自注意力是Transformer的基础——下一章的主角